In [1]:
"""
Manual symmetric int8 quantization of the frozen Fashion-MNIST MLP (ABACUS-6).

Weights:     per-tensor scale s_w = max(|W|) / 127, quantized to int8.
Activations: calibrated on ~1,000 training images, per-layer input max-abs,
             comparing plain max vs. 99.9th-percentile clipping.
Biases:      folded to int32 as round(b / (s_w * s_x)).

Loads the checkpoint produced by mlp_model.ipynb -- never retrains.
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

In [2]:
# ----------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------
SEED           = 0
DATA_DIR       = "./data"
CKPT_PATH      = "fmnist_mlp_fp32.pt"
CALIB_N        = 1000          # images used for activation calibration
PERCENTILE     = 99.9

DEVICE = torch.device("cpu")   # quantization sim runs fine on CPU

torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
# ----------------------------------------------------------------------
# Data (same normalization as training)
# ----------------------------------------------------------------------
MEAN, STD = 0.2860, 0.3530

tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
    transforms.Lambda(lambda x: x.view(-1)),
])

train_set = datasets.FashionMNIST(DATA_DIR, train=True,  download=True, transform=tfm)
test_set  = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=tfm)

test_loader = DataLoader(test_set, batch_size=512, shuffle=False, num_workers=0)

# Fixed, reproducible calibration subset
g = torch.Generator().manual_seed(SEED)
calib_idx = torch.randperm(len(train_set), generator=g)[:CALIB_N]
calib_set = Subset(train_set, calib_idx)
calib_loader = DataLoader(calib_set, batch_size=250, shuffle=False, num_workers=0)

In [4]:
# ----------------------------------------------------------------------
# Model + frozen checkpoint
# ----------------------------------------------------------------------
class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=512, n_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, n_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
model = MLP(**ckpt["arch"]).to(DEVICE)
model.load_state_dict(ckpt["state_dict"])
model.eval()

LAYERS = [model.fc1, model.fc2, model.fc3]
ckpt_acc = ckpt["test_acc"]
print(f"loaded checkpoint, fp32 test acc = {ckpt_acc*100:.2f}%")

loaded checkpoint, fp32 test acc = 89.51%


In [5]:
@torch.no_grad()
def evaluate_fp32(loader):
    correct, total = 0, 0
    for x, y in loader:
        logits = model(x)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total


fp32_acc = evaluate_fp32(test_loader)
print(f"reproduced fp32 test acc: {fp32_acc*100:.2f}%")

reproduced fp32 test acc: 89.51%


In [6]:
# ----------------------------------------------------------------------
# Weights: per-tensor symmetric int8, s_w = max(|W|) / 127
# ----------------------------------------------------------------------
def quantize_weight(w):
    s_w = w.abs().max().item() / 127.0
    q_w = torch.clamp(torch.round(w / s_w), -127, 127).to(torch.int8)
    return q_w, s_w


weight_q, weight_scale = {}, {}
for name, layer in zip(("fc1", "fc2", "fc3"), LAYERS):
    q_w, s_w = quantize_weight(layer.weight.data)
    weight_q[name], weight_scale[name] = q_w, s_w
    err = (q_w.float() * s_w - layer.weight.data).abs().max().item()
    print(f"{name}: s_w={s_w:.6e}  max|W|={layer.weight.data.abs().max().item():.4f}  "
          f"max dequant error={err:.6e}")

fc1: s_w=1.158645e-02  max|W|=1.4715  max dequant error=5.793217e-03
fc2: s_w=5.852339e-03  max|W|=0.7432  max dequant error=2.926171e-03
fc3: s_w=8.686239e-03  max|W|=1.1032  max dequant error=4.341662e-03


In [7]:
# ----------------------------------------------------------------------
# Activations: calibrate per-layer input max-abs over CALIB_N training images.
# Compare plain max vs. 99.9th-percentile clipping.
# ----------------------------------------------------------------------
captured = {"fc1": [], "fc2": [], "fc3": []}
hooks = []

def make_hook(name):
    def hook(module, inputs):
        captured[name].append(inputs[0].detach().abs().flatten())
    return hook

for name, layer in zip(("fc1", "fc2", "fc3"), LAYERS):
    hooks.append(layer.register_forward_pre_hook(make_hook(name)))

with torch.no_grad():
    for x, _ in calib_loader:
        model(x)

for h in hooks:
    h.remove()

act_max, act_p999 = {}, {}
for name in ("fc1", "fc2", "fc3"):
    vals = torch.cat(captured[name])
    act_max[name]  = vals.max().item()
    act_p999[name] = torch.quantile(vals, PERCENTILE / 100.0).item()
    print(f"{name}: max|x|={act_max[name]:.4f}  p{PERCENTILE}|x|={act_p999[name]:.4f}")

fc1: max|x|=2.0227  p99.9|x|=2.0227
fc2: max|x|=38.6543  p99.9|x|=17.2337
fc3: max|x|=90.8974  p99.9|x|=26.2136


In [8]:
# ----------------------------------------------------------------------
# Fake-quant forward pass: quantize activations + weights, fold bias,
# dequantize between layers. Used both to pick max vs. percentile and
# to validate the final int8 parameters end to end.
#
# act_bound is the calibrated max|x| (or percentile) -- the same kind of
# quantity as max|W| for weights. Same convention as weights: the actual
# per-step scale is that bound divided by 127, so that the bound itself
# maps to code +-127. (An earlier version of this notebook used act_bound
# directly as the scale, which under-used the int8 range by ~127x and
# collapsed accuracy to ~35% -- fixed here.)
# ----------------------------------------------------------------------
def build_bias_int32(s_w, s_x, bias):
    return torch.round(bias / (s_w * s_x)).to(torch.int32)


def quantized_forward(x, act_bound):
    for i, name in enumerate(("fc1", "fc2", "fc3")):
        layer = LAYERS[i]
        s_x = act_bound[name] / 127.0
        q_x = torch.clamp(torch.round(x / s_x), -127, 127)

        s_w = weight_scale[name]
        q_w = weight_q[name]
        b_i32 = build_bias_int32(s_w, s_x, layer.bias.data)

        acc = q_x @ q_w.float().T + b_i32.float()   # int32-equivalent accumulator
        x = acc * (s_w * s_x)                        # dequantize to fp32
        if i < 2:
            x = F.relu(x)
    return x


@torch.no_grad()
def evaluate_quantized(loader, act_bound):
    correct, total = 0, 0
    for x, y in loader:
        logits = quantized_forward(x, act_bound)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total

In [9]:
acc_with_max  = evaluate_quantized(test_loader, act_max)
acc_with_p999 = evaluate_quantized(test_loader, act_p999)

print(f"fp32 baseline:              {fp32_acc*100:.2f}%")
print(f"int8 sim, max calibration:  {acc_with_max*100:.2f}%  (drop {(fp32_acc-acc_with_max)*100:+.2f} pp)")
print(f"int8 sim, p999 calibration: {acc_with_p999*100:.2f}%  (drop {(fp32_acc-acc_with_p999)*100:+.2f} pp)")

if acc_with_p999 >= acc_with_max:
    act_bound, calib_method = act_p999, "p999"
else:
    act_bound, calib_method = act_max, "max"

print(f"\nselected calibration method: {calib_method}")

fp32 baseline:              89.51%
int8 sim, max calibration:  89.41%  (drop +0.10 pp)
int8 sim, p999 calibration: 89.43%  (drop +0.08 pp)

selected calibration method: p999


In [10]:
# ----------------------------------------------------------------------
# Final quantization parameters: int8 weights, per-layer scales,
# int32-folded biases -- using the selected activation calibration.
# ----------------------------------------------------------------------
act_scale = {name: act_bound[name] / 127.0 for name in ("fc1", "fc2", "fc3")}

bias_int32 = {
    name: build_bias_int32(weight_scale[name], act_scale[name], LAYERS[i].bias.data)
    for i, name in enumerate(("fc1", "fc2", "fc3"))
}

final_acc = evaluate_quantized(test_loader, act_bound)
print(f"final int8 model test acc: {final_acc*100:.2f}%  "
      f"(fp32 baseline {fp32_acc*100:.2f}%, drop {(fp32_acc-final_acc)*100:+.2f} pp)")

for name in ("fc1", "fc2", "fc3"):
    print(f"{name}: s_w={weight_scale[name]:.6e}  s_x={act_scale[name]:.6e}  "
          f"q_w range=[{weight_q[name].min()},{weight_q[name].max()}]  "
          f"b_int32 range=[{bias_int32[name].min()},{bias_int32[name].max()}]")

final int8 model test acc: 89.43%  (fp32 baseline 89.51%, drop +0.08 pp)
fc1: s_w=1.158645e-02  s_x=1.592648e-02  q_w range=[-127,90]  b_int32 range=[-1076,422]
fc2: s_w=5.852339e-03  s_x=1.356983e-01  q_w range=[-127,101]  b_int32 range=[-318,326]
fc3: s_w=8.686239e-03  s_x=2.064065e-01  q_w range=[-127,61]  b_int32 range=[-128,33]


In [11]:
# ----------------------------------------------------------------------
# Save quantization parameters for downstream use (golden NumPy model,
# ABACUS-8; requantize() unit tests, ABACUS-9).
# ----------------------------------------------------------------------
OUT_PATH = "quant_params_ptq.npz"

save_dict = {"calib_method": calib_method, "fp32_test_acc": fp32_acc, "int8_test_acc": final_acc}
for name in ("fc1", "fc2", "fc3"):
    save_dict[f"{name}_weight_int8"] = weight_q[name].numpy()
    save_dict[f"{name}_s_w"]         = weight_scale[name]
    save_dict[f"{name}_s_x"]         = act_scale[name]
    save_dict[f"{name}_bias_int32"]  = bias_int32[name].numpy()

np.savez(OUT_PATH, **save_dict)
print(f"saved -> {OUT_PATH}")

saved -> quant_params_ptq.npz


In [12]:
# ----------------------------------------------------------------------
# Finding: the first version of this notebook used the calibrated bound
# (max|x| or the percentile) directly as the per-step scale, instead of
# bound/127 the way the weight scale does (s_w = max|W| / 127). That
# under-used the int8 range by ~127x and collapsed accuracy to ~35%
# (best of max/p999) -- verified by reproducing the bug in isolation and
# comparing against the divide-by-127 fix, which alone recovers to
# ~89.4%, matching the fp32 baseline (89.51%) with no fine-tuning at all.
# The "root cause" analysis in that first version (i.i.d. quantization
# noise not canceling through correlated pixels; ~90%-sparse, long-tailed
# fc2/fc3 activations forcing an outlier-vs-resolution tradeoff) was
# describing real, measurable effects -- but they were secondary to this
# scale bug, not the primary cause, and the percentile sweep that seemed
# to confirm them was run entirely under the same bug.
# The activation scale formula is fixed above (see quantized_forward);
# this cell exists to record why the numbers below differ from that
# earlier (wrong) analysis.

In [13]:
# ========================================================================
# QAT: fine-tune the MLP with fake-quantized weights and activations so
# gradients can adapt the network to quantization noise, instead of
# hoping a fixed post-hoc calibration survives it. Requested on top of
# the corrected PTQ result above -- PTQ alone already recovers to ~89.4%
# once the scale bug is fixed, so QAT's job here is the smaller remaining
# gap plus removing the manual max-vs-p999 guesswork.
#
# Model tweak: ReLU -> learnable clipped-ReLU (PACT-style), one alpha per
# layer. alpha both replaces the nonlinearity AND sets the quant range
# (scale = alpha/127), so the clip/resolution tradeoff is learned end to
# end instead of chosen from two fixed calibration heuristics.
# Post-ReLU activations are never negative, so they're quantized to an
# unsigned 0..127 range -- 2x the resolution of the wasted-half symmetric
# range used for PTQ above. The raw pixel input keeps a learnable
# symmetric clip (it can be negative, no ReLU precedes it).
# Weights stay per-tensor symmetric int8 exactly as ABACUS-6 specs.
# ========================================================================
class RoundSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        return torch.round(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output


def fake_quantize(x, scale, qmin, qmax):
    x_clamped = torch.clamp(x / scale, qmin, qmax)
    return RoundSTE.apply(x_clamped) * scale


class QATMLP(nn.Module):
    def __init__(self, in_dim=784, hidden=512, n_classes=10,
                 init_input_clip=2.1, init_alpha=8.0):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, n_classes)
        self.input_clip_raw = nn.Parameter(torch.tensor(float(init_input_clip)))
        self.alpha1_raw = nn.Parameter(torch.tensor(float(init_alpha)))
        self.alpha2_raw = nn.Parameter(torch.tensor(float(init_alpha)))
        self.quant = True  # False -> plain fp32 forward, for A/B comparison

    @staticmethod
    def clipped_relu(x, alpha):
        return F.relu(x) - F.relu(x - alpha)

    def quant_weight(self, w):
        s_w = w.detach().abs().max() / 127.0
        return fake_quantize(w, s_w, -127, 127), s_w

    def forward(self, x):
        if not self.quant:
            h = F.relu(self.fc1(x))
            h = F.relu(self.fc2(h))
            return self.fc3(h)

        input_clip = F.softplus(self.input_clip_raw) + 1e-3
        alpha1 = F.softplus(self.alpha1_raw) + 1e-3
        alpha2 = F.softplus(self.alpha2_raw) + 1e-3

        x = torch.clamp(x, -input_clip, input_clip)
        x = fake_quantize(x, input_clip.detach() / 127.0, -127, 127)
        w1, _ = self.quant_weight(self.fc1.weight)
        h = self.clipped_relu(F.linear(x, w1, self.fc1.bias), alpha1)
        h = fake_quantize(h, alpha1.detach() / 127.0, 0, 127)

        w2, _ = self.quant_weight(self.fc2.weight)
        h = self.clipped_relu(F.linear(h, w2, self.fc2.bias), alpha2)
        h = fake_quantize(h, alpha2.detach() / 127.0, 0, 127)

        w3, _ = self.quant_weight(self.fc3.weight)
        return F.linear(h, w3, self.fc3.bias)

In [14]:
# ----------------------------------------------------------------------
# Fine-tune from the frozen fp32 checkpoint under fake quantization.
# ----------------------------------------------------------------------
QAT_EPOCHS = 10
QAT_LR = 3e-4
QAT_BATCH_SIZE = 128

train_loader = DataLoader(train_set, batch_size=QAT_BATCH_SIZE, shuffle=True, num_workers=0)

qat_model = QATMLP(**ckpt["arch"],
                    init_input_clip=act_max["fc1"],
                    init_alpha=max(act_p999["fc2"], act_p999["fc3"]))
qat_model.load_state_dict(model.state_dict(), strict=False)  # fc1/fc2/fc3 only; clip params keep their init
qat_model.quant = True

opt = torch.optim.Adam(qat_model.parameters(), lr=QAT_LR)
criterion = nn.CrossEntropyLoss()


@torch.no_grad()
def evaluate_qat(loader):
    qat_model.eval()
    correct, total = 0, 0
    for x, y in loader:
        logits = qat_model(x)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total


pre_ft_acc = evaluate_qat(test_loader)
print(f"QAT model before fine-tuning (fake-quant fwd, pretrained fp32 weights): {pre_ft_acc*100:.2f}%")

for epoch in range(1, QAT_EPOCHS + 1):
    qat_model.train()
    for x, y in train_loader:
        opt.zero_grad(set_to_none=True)
        loss = criterion(qat_model(x), y)
        loss.backward()
        opt.step()
    te_acc = evaluate_qat(test_loader)
    a1 = F.softplus(qat_model.alpha1_raw).item()
    a2 = F.softplus(qat_model.alpha2_raw).item()
    ic = F.softplus(qat_model.input_clip_raw).item()
    print(f"epoch {epoch:2d}/{QAT_EPOCHS} | test acc {te_acc*100:.2f}% | "
          f"input_clip={ic:.3f} alpha1={a1:.3f} alpha2={a2:.3f}")

QAT model before fine-tuning (fake-quant fwd, pretrained fp32 weights): 89.54%


epoch  1/10 | test acc 89.99% | input_clip=2.147 alpha1=26.229 alpha2=26.203


epoch  2/10 | test acc 90.36% | input_clip=2.147 alpha1=26.229 alpha2=26.211


epoch  3/10 | test acc 90.20% | input_clip=2.147 alpha1=26.229 alpha2=26.211


epoch  4/10 | test acc 90.10% | input_clip=2.147 alpha1=26.229 alpha2=26.212


...

epoch  6/10 | test acc 90.28% | input_clip=2.147 alpha1=26.229 alpha2=26.212


epoch  7/10 | test acc 90.02% | input_clip=2.147 alpha1=26.229 alpha2=26.212


epoch  8/10 | test acc 90.13% | input_clip=2.147 alpha1=26.229 alpha2=26.212


epoch  9/10 | test acc 90.10% | input_clip=2.147 alpha1=26.229 alpha2=26.212


epoch 10/10 | test acc 90.25% | input_clip=2.147 alpha1=26.229 alpha2=26.212


In [15]:
# ----------------------------------------------------------------------
# Export final QAT int8 parameters: weights + scales from the fine-tuned
# model, biases folded to int32 exactly as in the PTQ path above -- only
# the activation quant range (learned alpha/input_clip, and unsigned
# range for post-ReLU layers) differs from the PTQ export.
# ----------------------------------------------------------------------
qat_model.eval()
with torch.no_grad():
    qat_input_clip = (F.softplus(qat_model.input_clip_raw) + 1e-3).item()
    qat_alpha1 = (F.softplus(qat_model.alpha1_raw) + 1e-3).item()
    qat_alpha2 = (F.softplus(qat_model.alpha2_raw) + 1e-3).item()

qat_act_scale = {"fc1": qat_input_clip / 127.0, "fc2": qat_alpha1 / 127.0, "fc3": qat_alpha2 / 127.0}
qat_qmin = {"fc1": -127, "fc2": 0, "fc3": 0}

qat_weight_q, qat_weight_scale = {}, {}
for name, layer in zip(("fc1", "fc2", "fc3"), (qat_model.fc1, qat_model.fc2, qat_model.fc3)):
    q_w, s_w = quantize_weight(layer.weight.data)
    qat_weight_q[name], qat_weight_scale[name] = q_w, s_w

qat_bias_int32 = {
    name: build_bias_int32(qat_weight_scale[name], qat_act_scale[name], layer.bias.data)
    for name, layer in zip(("fc1", "fc2", "fc3"), (qat_model.fc1, qat_model.fc2, qat_model.fc3))
}


def qat_quantized_forward(x):
    h = x
    for name, qmin in zip(("fc1", "fc2", "fc3"), (-127, 0, 0)):
        s_x = qat_act_scale[name]
        q_h = torch.clamp(torch.round(h / s_x), qmin, 127)
        s_w = qat_weight_scale[name]
        acc = q_h @ qat_weight_q[name].float().T + qat_bias_int32[name].float()
        h = acc * (s_w * s_x)
        if name != "fc3":
            h = torch.clamp(h, min=0.0)  # clipped-relu already bounds the pre-quant range
    return h


@torch.no_grad()
def evaluate_qat_int8(loader):
    correct, total = 0, 0
    for x, y in loader:
        logits = qat_quantized_forward(x)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total


qat_int8_acc = evaluate_qat_int8(test_loader)
print(f"fp32 baseline:                {fp32_acc*100:.2f}%")
print(f"PTQ int8 (best of max/p999):  {final_acc*100:.2f}%")
print(f"QAT int8 (fine-tuned):        {qat_int8_acc*100:.2f}%  "
      f"(drop {(fp32_acc-qat_int8_acc)*100:+.2f} pp vs fp32)")
for name in ("fc1", "fc2", "fc3"):
    print(f"{name}: s_w={qat_weight_scale[name]:.6e}  s_x={qat_act_scale[name]:.6e}  "
          f"q_w range=[{qat_weight_q[name].min()},{qat_weight_q[name].max()}]  "
          f"b_int32 range=[{qat_bias_int32[name].min()},{qat_bias_int32[name].max()}]")

fp32 baseline:                89.51%
PTQ int8 (best of max/p999):  89.43%
QAT int8 (fine-tuned):        90.24%  (drop -0.73 pp vs fp32)
fc1: s_w=1.165706e-02  s_x=1.691273e-02  q_w range=[-127,90]  b_int32 range=[-1007,396]
fc2: s_w=5.825904e-03  s_x=2.065324e-01  q_w range=[-127,104]  b_int32 range=[-217,221]
fc3: s_w=8.686239e-03  s_x=2.063989e-01  q_w range=[-127,62]  b_int32 range=[-133,35]


In [16]:
# ----------------------------------------------------------------------
# Save the QAT-tuned fp32 checkpoint and the recommended int8 deployment
# parameters (this is what ABACUS-8/9 should consume, not the PTQ file).
# ----------------------------------------------------------------------
torch.save({
    "state_dict": qat_model.state_dict(),
    "arch": ckpt["arch"],
    "test_acc_fake_quant": qat_int8_acc,
}, "fmnist_mlp_qat.pt")

QAT_OUT_PATH = "quant_params_qat.npz"
qat_save_dict = {"fp32_test_acc": fp32_acc, "int8_test_acc": qat_int8_acc}
for name in ("fc1", "fc2", "fc3"):
    qat_save_dict[f"{name}_weight_int8"] = qat_weight_q[name].numpy()
    qat_save_dict[f"{name}_s_w"]         = qat_weight_scale[name]
    qat_save_dict[f"{name}_s_x"]         = qat_act_scale[name]
    qat_save_dict[f"{name}_qmin"]        = qat_qmin[name]
    qat_save_dict[f"{name}_bias_int32"]  = qat_bias_int32[name].numpy()

np.savez(QAT_OUT_PATH, **qat_save_dict)
print(f"saved -> fmnist_mlp_qat.pt, {QAT_OUT_PATH}")

saved -> fmnist_mlp_qat.pt, quant_params_qat.npz
